# Demo 02 : Reducers

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · add-messages

We port ARI510 demo20's message edits. add_messages appends new ids, replaces an existing id and applies RemoveMessage.

In [2]:
from langchain.messages import AIMessage, HumanMessage, RemoveMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages

history = add_messages([], [HumanMessage("Inspect bus 4012", id="request"), AIMessage("Pending", id="reply")])
history = add_messages(history, [AIMessage("Assigned to workshop", id="reply")])
print("edited :", [(message.id, message.content) for message in history])
history = add_messages(history, [RemoveMessage(id="request")])
print("removed :", [(message.id, message.content) for message in history])

class ServiceState(MessagesState):
    depot: str

messages = StateGraph(ServiceState)
messages.add_node("reply", lambda state: {"messages": [AIMessage("Workshop notified")]})
messages.add_edge(START, "reply")
messages.add_edge("reply", END)
result = messages.compile().invoke({"messages": [("user", "Inspect bus 4012")], "depot": "Noord"})
print("messages :", [message.content for message in result["messages"]])
print("depot :", result["depot"])

edited : [('request', 'Inspect bus 4012'), ('reply', 'Assigned to workshop')]
removed : [('reply', 'Assigned to workshop')]
messages : ['Inspect bus 4012', 'Workshop notified']
depot : Noord


## b · custom-reducers

We first keep ARI510 demo20's small sequential example. We then merge workshop findings with a custom reducer that removes duplicates and sorts the result.

In [3]:
import operator
from typing import Annotated, TypedDict

class Collect(TypedDict):
    collected: Annotated[list[str], operator.add]
    last: str

collected = StateGraph(Collect)
collected.add_node("a", lambda state: {"collected": ["a"], "last": "a"})
collected.add_node("b", lambda state: {"collected": ["b"], "last": "b"})
collected.add_edge(START, "a")
collected.add_edge("a", "b")
collected.add_edge("b", END)
print("sequential :", collected.compile().invoke({"collected": [], "last": ""}))

def merge_findings(current: list[str], incoming: list[str]) -> list[str]:
    return sorted(set(current) | set(incoming))

class Findings(TypedDict):
    findings: Annotated[list[str], merge_findings]

findings = StateGraph(Findings)
findings.add_node("inspect", lambda state: {"findings": ["brake inspection", "brake inspection"]})
findings.add_edge(START, "inspect")
findings.add_edge("inspect", END)
print("merged :", findings.compile().invoke({"findings": ["parts lookup"]}))

sequential : {'collected': ['a', 'b'], 'last': 'b'}
merged : {'findings': ['brake inspection', 'parts lookup']}


## c · overwriting

We replace a reduced value with Overwrite. An ordinary empty list would pass through the reducer and retain the existing findings.

In [4]:
from langgraph.types import Overwrite

reset = StateGraph(Findings)
reset.add_node("clear", lambda state: {"findings": Overwrite(value=[])})
reset.add_edge(START, "clear")
reset.add_edge("clear", END)
print("ordinary empty update :", merge_findings(["parts lookup"], []))
print("overwrite :", reset.compile().invoke({"findings": ["parts lookup"]}))

ordinary empty update : ['parts lookup']
overwrite : {'findings': []}


An `Overwrite` and an ordinary write to the same key can meet in one superstep. Branch `a` replaces the findings while branch `b` adds one.

In [5]:
race = StateGraph(Findings)
race.add_node("a", lambda state: {"findings": Overwrite(["brake inspection"])})
race.add_node("b", lambda state: {"findings": ["parts lookup"]})
race.add_edge(START, "a")
race.add_edge(START, "b")
race.add_edge(["a", "b"], END)
print("overwrite and write :", race.compile().invoke({"findings": ["ticket opened"]}))

overwrite and write : {'findings': ['brake inspection']}


The ordinary write from `b` is gone, and no error is raised. Two `Overwrite` values for one key in one step do raise `InvalidUpdateError`.

## d · concurrent-updates

We run two parallel branches that write one unreduced key. We catch InvalidUpdateError, then keep both findings with a reducer.

In [6]:
from langgraph.errors import InvalidUpdateError

class NoReducer(TypedDict):
    result: str

parallel = StateGraph(NoReducer)
parallel.add_node("parts", lambda state: {"result": "parts lookup"})
parallel.add_node("workshop", lambda state: {"result": "brake inspection"})
parallel.add_edge(START, "parts")
parallel.add_edge(START, "workshop")
parallel.add_edge("parts", END)
parallel.add_edge("workshop", END)
try:
    parallel.compile().invoke({"result": ""})
except InvalidUpdateError as error:
    print("InvalidUpdateError :", str(error).splitlines()[0])

fixed = StateGraph(Findings)
fixed.add_node("parts", lambda state: {"findings": ["parts lookup"]})
fixed.add_node("workshop", lambda state: {"findings": ["brake inspection"]})
fixed.add_edge(START, "parts")
fixed.add_edge(START, "workshop")
fixed.add_edge("parts", END)
fixed.add_edge("workshop", END)
print("fixed :", fixed.compile().invoke({"findings": []}))

InvalidUpdateError : At key 'result': Can receive only one value per step. Use an Annotated key to handle multiple values.
fixed : {'findings': ['brake inspection', 'parts lookup']}


## Things to try

Optional : we can change a ticket, a thread id or a returned update and compare the saved state.